# Rio Controller - Interactive Control Interface

This notebook provides an interactive UI for controlling the Rio microfluidics controller using ipywidgets.

**Note:** The API now uses **LabThings/WoT-compliant Things**. This notebook uses the legacy `/api/control/*` routes for backward compatibility. WoT routes are available at `/flow/`, `/heater/`, `/camera/`, etc. (see API docs at `/docs`).

## Features

- **Flow/Pressure Control**: Set flow rates and pressures with sliders
- **Heater Control**: Set temperatures and enable PID/stirrer
- **Camera**: Live stream and snapshot capture
- **Strobe**: Control strobe timing and enable/disable
- **Syringe Pump**: Flow, diameter, direction, start/stop, and advanced settings
- **Real-time Status**: Live updates of sensor readings
- **Emergency Stop**: Quick stop for all operations

## Prerequisites

1. Rio API server must be running (see `software/api/README.md`)
2. Install required packages:
   ```bash
   pip install requests websocket-client ipywidgets matplotlib pandas numpy
   ```


In [ ]:
# Configuration and imports
import os
import sys
from pathlib import Path
import base64
import time
from datetime import datetime
from IPython.display import display, Image, HTML, clear_output
import ipywidgets as widgets

# Add software/api/client to path
repo_root = Path.cwd()
if (repo_root / "software" / "api" / "client").exists():
    sys.path.insert(0, str(repo_root / "software"))
elif (repo_root.parent.parent / "software" / "api" / "client").exists():
    sys.path.insert(0, str(repo_root.parent.parent))
else:
    for parent in repo_root.parents:
        if (parent / "software" / "api" / "client").exists():
            sys.path.insert(0, str(parent / "software"))
            break

from api.client import RioClient, RioAPIError
from api.client.syringe_pump_api import PumpAPIError, SyringePumpAPI

# API configuration
API_BASE_URL = os.getenv("RIO_API_URL", "http://localhost:8000")
PUMP_API_URL = os.getenv("RIO_PUMP_API_URL", API_BASE_URL)

# Initialize client
try:
    client = RioClient(base_url=API_BASE_URL)
    health = client.health()
    print(f"✅ Connected to Rio API at {API_BASE_URL}")
    print(f"   Status: {health['status']}, Simulation: {health['simulation']}")
except Exception as e:
    print(f"❌ Failed to connect to API: {e}")
    print(f"   Make sure the API server is running at {API_BASE_URL}")
    raise

# Initialize pump API (lazy connection)
try:
    pump_api = SyringePumpAPI(base_url=PUMP_API_URL)
    print(f"✅ Pump API configured at {PUMP_API_URL}")
except Exception as e:
    pump_api = None
    print(f"⚠️ Pump API unavailable: {e}")


# Main UI - Tabbed Interface

The UI is organized into tabs for easy navigation, similar to the internship interface.


In [6]:
# Configuration
import os
import sys
from pathlib import Path
import requests
import json
import time
from datetime import datetime
from IPython.display import display, Image, HTML, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Set API base URL (can also use environment variable RIO_API_URL)
# For local development: "http://localhost:8000"
# For Raspberry Pi: "http://raspberrypi.local:8000" or "http://192.168.1.100:8000"
API_BASE_URL = os.getenv("RIO_API_URL", "http://localhost:8000")
# Uncomment and update for Pi access:
# API_BASE_URL = "http://raspberrypi.local:8000"

HERE = Path.cwd().resolve()

PROJECT_ROOT = None
for parent in [HERE] + list(HERE.parents):
    if (parent / "api" / "client").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "❌ Could not find project root containing api/client.\n"
        f"Current directory: {HERE}"
    )

sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root detected:", PROJECT_ROOT)
print("📦 sys.path[0]:", sys.path[0])

# -------------------------
# Imports
# -------------------------
from api.client.api_client import RioClient, RioStreamClient
from api.client.syringe_pump_api import SyringePumpAPI, PumpAPIError

# -------------------------
# Initialize clients
# -------------------------
client = RioClient(base_url=API_BASE_URL)
pump_api = SyringePumpAPI(base_url=API_BASE_URL, use_wot=False)

print("✅ Rio API client initialized")
print(f"   Base URL: {API_BASE_URL}")

✅ Project root detected: /home/pi/pi-deployment
📦 sys.path[0]: /home/pi/pi-deployment
✅ Rio API client initialized
   Base URL: http://localhost:8000


In [5]:
# Create main UI with tabs (inspired by internship interface)
def create_ui():
    # Create tab container
    tab = widgets.Tab()
    status_output = widgets.Output()
    
    # ===== FLOW/PRESSURE TAB =====
    # Use Dropdown for channel selection (like internship uses Dropdown for pump selection)
    flow_channel = widgets.Dropdown(options=[0, 1, 2, 3], value=0, description='Channel:', style={'description_width': 'initial'})
    flow_rate = widgets.FloatSlider(value=0, min=0, max=1000, step=1, description='Flow (ul/hr):', style={'description_width': 'initial'})
    pressure_value = widgets.FloatSlider(value=0, min=0, max=200, step=1, description='Pressure (mbar):', style={'description_width': 'initial'})
    flow_status = widgets.Output(layout=widgets.Layout(
        width='350px',
        height='200px',
        overflow_y='auto',
        border='1px solid #ddd',
        padding='10px',
        margin='10px 0'
    ))
    
    def update_flow_state():
        """Update flow state display"""
        try:
            state = client.get_flow_state()
            with flow_status:
                clear_output()
                print("Current Flow/Pressure State:")
                for i in range(4):
                    print(f"\n  Channel {i}:")
                    print(f"    Flow: {state['flow_actuals_ul_hr'][i]:.1f} ul/hr (target: {state['flow_targets_ul_hr'][i]:.1f})")
                    print(f"    Pressure: {state['pressure_actuals_mbar'][i]:.1f} mbar (target: {state['pressure_targets_mbar'][i]:.1f})")
                    print(f"    Mode: {state['control_modes_text'][i]}")
        except Exception as e:
            with flow_status:
                clear_output()
                print(f"❌ Error: {e}")
    
    def create_flow_handler(param_name, setter_func):
        """Create parameter handler for flow/pressure (like internship pattern)"""
        def handler(change):
            if change['name'] == 'value':
                try:
                    channel = flow_channel.value
                    value = change['new']
                    getattr(client, setter_func)(channel, value)
                    with flow_status:
                        clear_output()
                        print(f"✅ Updated {param_name} for channel {channel} to {value}")
                    update_flow_state()
                except Exception as e:
                    with flow_status:
                        clear_output()
                        print(f"❌ Error updating {param_name}: {e}")
        return handler
    
    def on_channel_select(change):
        """Handle channel selection change - load current values"""
        if change['name'] == 'value':
            try:
                state = client.get_flow_state()
                channel = change['new']
                flow_rate.value = state['flow_targets_ul_hr'][channel]
                pressure_value.value = state['pressure_targets_mbar'][channel]
                update_flow_state()
            except Exception as e:
                with flow_status:
                    print(f"❌ Error loading channel {change['new']}: {e}")
    
    flow_channel.observe(on_channel_select, names='value')
    flow_rate.observe(create_flow_handler('flow', 'set_flow'), 'value')
    pressure_value.observe(create_flow_handler('pressure', 'set_pressure'), 'value')
    
    flow_basic = widgets.VBox([
        widgets.HTML("<h3>Basic Controls</h3>"),
        flow_channel,
        flow_rate,
        pressure_value
    ], layout=widgets.Layout(margin='0 20px 0 0'))
    
    flow_status_box = widgets.VBox([
        widgets.HTML("<h3>Status</h3>"),
        widgets.Button(description='🔄 Refresh', button_style='info'),
        flow_status
    ], layout=widgets.Layout(min_width='350px', margin='0 0 0 20px'))
    
    flow_tab = widgets.HBox([flow_basic, flow_status_box])
    flow_tab.children[1].children[1].on_click(lambda b: update_flow_state())
    
    # ===== HEATER TAB =====
    # Use Dropdown for heater selection
    heater_select = widgets.Dropdown(options=[0, 1, 2, 3], value=0, description='Heater:', style={'description_width': 'initial'})
    heater_temp = widgets.FloatSlider(value=25, min=0, max=100, step=0.1, description='Temp (°C):', style={'description_width': 'initial'})
    heater_pid = widgets.ToggleButton(value=False, description='PID Control', button_style='info')
    heater_stir = widgets.ToggleButton(value=False, description='Stirrer', button_style='info')
    heater_status = widgets.Output(layout=widgets.Layout(
        width='350px',
        height='200px',
        overflow_y='auto',
        border='1px solid #ddd',
        padding='10px',
        margin='10px 0'
    ))
    
    def update_heater_state():
        """Update heater state display"""
        try:
            state = client.get_heater_state()
            with heater_status:
                clear_output()
                print("Current Heater States:")
                for i, h in enumerate(state['heaters']):
                    print(f"\n  Heater {i}:")
                    print(f"    Temp: {h['temp_c_actual']:.1f}°C (target: {h['temp_c_target']:.1f}°C)")
                    print(f"    PID: {'ON' if h['pid_enabled'] else 'OFF'}")
                    print(f"    Stir: {'ON' if h['stir_enabled'] else 'OFF'}")
                    print(f"    Status: {h['status_text']}")
        except Exception as e:
            with heater_status:
                clear_output()
                print(f"❌ Error: {e}")
    
    def create_heater_handler(param_name, setter_func):
        """Create parameter handler for heater controls"""
        def handler(change):
            if change['name'] == 'value':
                try:
                    heater = heater_select.value
                    value = change['new']
                    getattr(client, setter_func)(heater, value)
                    with heater_status:
                        clear_output()
                        print(f"✅ Updated {param_name} for heater {heater} to {value}")
                    update_heater_state()
                except Exception as e:
                    with heater_status:
                        clear_output()
                        print(f"❌ Error updating {param_name}: {e}")
        return handler
    
    def on_heater_select(change):
        """Handle heater selection change - load current values"""
        if change['name'] == 'value':
            try:
                state = client.get_heater_state()
                heater = change['new']
                h = state['heaters'][heater]
                heater_temp.value = h['temp_c_target']
                heater_pid.value = h['pid_enabled']
                heater_stir.value = h['stir_enabled']
                update_heater_state()
            except Exception as e:
                with heater_status:
                    print(f"❌ Error loading heater {change['new']}: {e}")
    
    heater_select.observe(on_heater_select, names='value')
    heater_temp.observe(create_heater_handler('temperature', 'set_heater_temp'), 'value')
    heater_pid.observe(create_heater_handler('PID', 'set_heater_pid'), 'value')
    heater_stir.observe(create_heater_handler('stirrer', 'set_heater_stir'), 'value')
    
    heater_basic = widgets.VBox([
        widgets.HTML("<h3>Basic Controls</h3>"),
        heater_select,
        heater_temp,
        widgets.HBox([heater_pid, heater_stir])
    ], layout=widgets.Layout(margin='0 20px 0 0'))
    
    heater_status_box = widgets.VBox([
        widgets.HTML("<h3>Status</h3>"),
        widgets.Button(description='🔄 Refresh', button_style='info'),
        heater_status
    ], layout=widgets.Layout(min_width='350px', margin='0 0 0 20px'))
    
    heater_tab = widgets.HBox([heater_basic, heater_status_box])
    heater_tab.children[1].children[1].on_click(lambda b: update_heater_state())
    
    # ===== CAMERA/STROBE TAB =====
    stream_btn = widgets.ToggleButton(value=False, description='▶ Start Camera', button_style='success')
    capture_btn = widgets.Button(description='📸 Capture Image', button_style='success')
    save_container = widgets.Output()
    enable_btn = widgets.ToggleButton(value=False, description='⏻ Enable Strobe', button_style='success')
    period = widgets.FloatSlider(min=1, max=10000, step=1, value=50, description='Period (µs):', style={'description_width': 'initial'})
    width = widgets.FloatSlider(min=0.1, max=1000, step=0.1, value=0.1, description='Width (µs):', style={'description_width': 'initial'})
    hold_btn = widgets.ToggleButton(value=False, description='🔆 Hold Mode')
    stream_container = widgets.Output(layout=widgets.Layout(width='400px', height='300px'))
    captured_container = widgets.Output(layout=widgets.Layout(width='400px', height='300px'))
    
    def toggle_stream(change):
        if change['name'] == 'value':
            if change['new']:
                stream_btn.description = '⏹ Stop Camera'
                stream_btn.button_style = 'danger'
                with stream_container:
                    clear_output()
                    # Note: MJPEG stream endpoint would go here if available
                    display(HTML('<div style="width:400px; height:300px; border:2px solid #4CAF50; border-radius:4px; display:flex; align-items:center; justify-content:center;">Camera Stream (MJPEG endpoint not yet available)</div>'))
            else:
                stream_btn.description = '▶ Start Camera'
                stream_btn.button_style = 'success'
                with stream_container:
                    clear_output()
                    display(HTML('<div style="width:400px; height:300px; border:2px dashed #ccc; display:flex; align-items:center; justify-content:center;">Stream Stopped</div>'))
    
    def capture_image(btn):
        try:
            snapshot = client.get_camera_snapshot()
            if snapshot:
                with captured_container:
                    clear_output()
                    display(Image(data=snapshot, width=400))
                with save_container:
                    clear_output()
                    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                    display(HTML(f"""
                        <a href="data:image/jpeg;base64,{base64.b64encode(snapshot).decode('utf-8')}"
                           download="capture_{timestamp}.jpg"
                           style="padding: 6px 12px; background-color: #4CAF50; color: white; 
                                  text-decoration: none; border-radius: 4px; display: inline-block;">
                            💾 Save Image
                        </a>
                    """))
                with status_output:
                    print("✅ Image captured! Click 'Save Image' to download.")
            else:
                with status_output:
                    print("❌ No image data received")
        except Exception as e:
            with status_output:
                print(f"❌ Error capturing image: {e}")
    
    def update_strobe(change=None):
        try:
            client.set_strobe_enable(enable_btn.value)
            if period.value:
                period_ns = int(period.value * 1000)
                width_ns = int(width.value * 1000)
                client.set_strobe_timing(period_ns, width_ns)
            client.set_strobe_hold(hold_btn.value)
        except Exception as e:
            with status_output:
                print(f"❌ Error updating strobe: {e}")
    
    def on_strobe_toggle(change):
        if change['name'] == 'value':
            if change['new']:
                enable_btn.description = '⏹ Disable Strobe'
                enable_btn.button_style = 'danger'
            else:
                enable_btn.description = '⏻ Enable Strobe'
                enable_btn.button_style = 'success'
        update_strobe()
    
    stream_btn.observe(toggle_stream, 'value')
    capture_btn.on_click(capture_image)
    enable_btn.observe(on_strobe_toggle, 'value')
    period.observe(update_strobe, 'value')
    width.observe(update_strobe, 'value')
    hold_btn.observe(update_strobe, 'value')
    
    camera_controls = widgets.HBox([stream_btn, capture_btn, save_container])
    strobe_controls = widgets.VBox([
        widgets.HBox([enable_btn, hold_btn]),
        period,
        width
    ])
    image_display = widgets.HBox([
        widgets.VBox([widgets.Label('Live Stream'), stream_container]),
        widgets.VBox([widgets.Label('Captured Image'), captured_container])
    ])
    
    camera_tab = widgets.VBox([
        widgets.HTML("<h3>Camera Controls</h3>"),
        camera_controls,
        widgets.HTML("<h3>Strobe Controls</h3>"),
        strobe_controls,
        widgets.HTML("<hr>"),
        image_display
    ])

    # ===== PUMP TAB =====
    pump_select = widgets.Dropdown(
        options=["A", "B", "C", "D"],
        value="A",
        description="Pump:",
        style={"description_width": "initial"},
    )
    pump_flow = widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=2000.0,
        step=1.0,
        description="Flow:",
        style={"description_width": "initial"},
    )
    pump_diameter = widgets.FloatSlider(
        value=5.0,
        min=0.1,
        max=50.0,
        step=0.1,
        description="Diameter (mm):",
        style={"description_width": "initial"},
    )
    pump_direction = widgets.Dropdown(
        options=["infuse", "withdraw"],
        value="infuse",
        description="Direction:",
        style={"description_width": "initial"},
    )
    pump_state = widgets.ToggleButton(
        value=False,
        description="▶ Run",
        button_style="success",
    )
    pump_unit = widgets.Dropdown(
        options=["UL/MIN", "UL/HR", "ML/MIN", "ML/HR"],
        value="UL/HR",
        description="Unit:",
        style={"description_width": "initial"},
    )
    pump_gearbox = widgets.Dropdown(
        options=["1:1", "25:1", "100:1"],
        value="1:1",
        description="Gearbox:",
        style={"description_width": "initial"},
    )
    pump_microstep = widgets.Dropdown(
        options=["1/8", "1/16", "1/32", "1/64"],
        value="1/16",
        description="Microstep:",
        style={"description_width": "initial"},
    )
    pump_threadrod = widgets.Dropdown(
        options=["1-START", "4-START"],
        value="1-START",
        description="Thread Rod:",
        style={"description_width": "initial"},
    )
    pump_enable = widgets.ToggleButton(
        value=True,
        description="Enabled",
        button_style="success",
    )
    pump_status = widgets.Output(
        layout=widgets.Layout(
            width="350px",
            height="200px",
            overflow_y="auto",
            border="1px solid #ddd",
            padding="10px",
            margin="10px 0",
        )
    )

    pump_refreshing = {"active": False}

    def refresh_pump_state():
        with pump_status:
            clear_output()
            if pump_api is None:
                print("⚠️ Pump API not configured")
                return
            try:
                pump_refreshing["active"] = True
                state = pump_api.get_state(pump_select.value)
                print(f"Pump {pump_select.value} state:")
                print(state)
                if isinstance(state, dict):
                    if state.get("flow") is not None:
                        pump_flow.value = float(state["flow"])
                    if state.get("diameter") is not None:
                        pump_diameter.value = float(state["diameter"])
                    if state.get("direction") is not None:
                        pump_direction.value = "infuse" if int(state["direction"]) >= 0 else "withdraw"
                    if state.get("state") is not None:
                        pump_state.value = bool(state["state"])
                        pump_state.description = "⏹ Stop" if pump_state.value else "▶ Run"
                        pump_state.button_style = "danger" if pump_state.value else "success"
                    if state.get("unit"):
                        pump_unit.value = state["unit"]
                    if state.get("gearbox"):
                        pump_gearbox.value = state["gearbox"]
                    if state.get("microstep"):
                        pump_microstep.value = state["microstep"]
                    if state.get("threadrod"):
                        pump_threadrod.value = state["threadrod"]
                    if state.get("enabled") is not None:
                        pump_enable.value = bool(state["enabled"])
                        pump_enable.description = "Enabled" if pump_enable.value else "Disabled"
                        pump_enable.button_style = "success" if pump_enable.value else "warning"
            except Exception as e:
                print(f"❌ Pump state error: {e}")
            finally:
                pump_refreshing["active"] = False

    def on_pump_flow(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_flow(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set flow: {e}")

    def on_pump_diameter(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_diameter(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set diameter: {e}")

    def on_pump_direction(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_direction(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set direction: {e}")

    def on_pump_state(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        pump_state.description = "⏹ Stop" if change["new"] else "▶ Run"
        pump_state.button_style = "danger" if change["new"] else "success"
        if pump_api is None:
            return
        try:
            pump_api.set_state(pump_select.value, "run" if change["new"] else "stop")
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set state: {e}")

    def on_pump_unit(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_unit(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set unit: {e}")

    def on_pump_gearbox(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_gearbox(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set gearbox: {e}")

    def on_pump_microstep(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_microstep(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set microstep: {e}")

    def on_pump_threadrod(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        if pump_api is None:
            return
        try:
            pump_api.set_threadrod(pump_select.value, change["new"])
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set thread rod: {e}")

    def on_pump_enable(change):
        if change["name"] != "value" or pump_refreshing["active"]:
            return
        pump_enable.description = "Enabled" if change["new"] else "Disabled"
        pump_enable.button_style = "success" if change["new"] else "warning"
        if pump_api is None:
            return
        try:
            pump_api.set_enable(pump_select.value, bool(change["new"]))
            refresh_pump_state()
        except Exception as e:
            with pump_status:
                clear_output()
                print(f"❌ Failed to set enable: {e}")

    pump_select.observe(lambda change: refresh_pump_state(), names="value")
    pump_flow.observe(on_pump_flow, names="value")
    pump_diameter.observe(on_pump_diameter, names="value")
    pump_direction.observe(on_pump_direction, names="value")
    pump_state.observe(on_pump_state, names="value")
    pump_unit.observe(on_pump_unit, names="value")
    pump_gearbox.observe(on_pump_gearbox, names="value")
    pump_microstep.observe(on_pump_microstep, names="value")
    pump_threadrod.observe(on_pump_threadrod, names="value")
    pump_enable.observe(on_pump_enable, names="value")

    pump_controls = widgets.VBox(
        [
            widgets.HTML("<h3>Basic Controls</h3>"),
            pump_select,
            pump_flow,
            pump_diameter,
            pump_direction,
            pump_state,
            pump_unit,
            widgets.HTML("<h3>Advanced Settings</h3>"),
            pump_gearbox,
            pump_microstep,
            pump_threadrod,
            pump_enable,
            widgets.Button(description="🔄 Refresh", button_style="info"),
        ]
    )
    pump_controls.children[-1].on_click(lambda b: refresh_pump_state())

    pump_tab = widgets.HBox([
        pump_controls,
        pump_status,
    ])

    # ===== EMERGENCY STOP =====
    emergency_btn = widgets.Button(
        description='EMERGENCY STOP',
        button_style='warning',
        icon='exclamation-triangle',
        layout=widgets.Layout(width='300px', height='50px')
    )
    emergency_status = widgets.Output(layout=widgets.Layout(height='100px', overflow_y='auto', border='1px solid #ddd', padding='10px'))
    
    def emergency_stop(btn):
        try:
            # Stop all flow channels
            for i in range(4):
                try:
                    client.set_flow(i, 0.0)
                    client.set_pressure(i, 0.0)
                except Exception as e:
                    pass
            
            # Disable all heaters
            for i in range(4):
                try:
                    client.set_heater_pid(i, False)
                    client.set_heater_stir(i, False)
                except Exception as e:
                    pass
            
            # Disable strobe
            try:
                client.set_strobe_enable(False)
            except Exception as e:
                pass
            
            with emergency_status:
                clear_output()
                print("🛑 EMERGENCY STOP: All operations stopped")
                print("   - All flow channels set to 0")
                print("   - All heaters disabled")
                print("   - Strobe disabled")
        except Exception as e:
            with emergency_status:
                print(f"❌ Emergency stop error: {e}")
    
    emergency_btn.on_click(emergency_stop)
    
    # Set up tabs
    tab.children = [flow_tab, heater_tab, camera_tab, pump_tab]
    tab.titles = ['Flow/Pressure', 'Heater', 'Camera/Strobe', 'Pump']
    
    # Display UI
    display(tab)
    display(widgets.VBox([emergency_btn, emergency_status]))
    display(status_output)
    
    # Initialize containers
    with stream_container:
        display(HTML('<div style="width:400px; height:300px; border:2px dashed #ccc; display:flex; align-items:center; justify-content:center;">Stream Stopped</div>'))
    with captured_container:
        display(HTML('<div style="width:400px; height:300px; border:2px dashed #ccc; display:flex; align-items:center; justify-content:center;">No Image Captured</div>'))
    
    # Initial state updates
    update_flow_state()
    update_heater_state()
    refresh_pump_state()
    
    # Return the tab for potential reuse
    return tab

# Start the UI
create_ui()


Output()